### Transform data to 28 x 28

In [44]:
from torchvision import datasets, transforms
import torch

# Data transformations
transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
])

# Load the dataset
train_set = datasets.ImageFolder(root="../../dataset/train", transform=transform)
test_set = datasets.ImageFolder(root='../../dataset/test', transform=transform)

train_loader = torch.utils.data.DataLoader(train_set, batch_size=16, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=16, shuffle=False)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# Define the simple CNN
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # Input channels = 3 (RGB), output channels = 16
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 16, kernel_size=3)
        self.flat = nn.Flatten()
        self.fc1 = nn.Linear(in_features=26*26*16, out_features=128)
        #self.drop = nn.Dropout(0.25)
        self.fc2 = nn.Linear(in_features=128, out_features=64)
        self.fc3 = nn.Linear(in_features=64, out_features=6)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.flat(x)
        x = F.relu(self.fc1(x))
        #x = self.drop(x)
        x = self.fc2(x)
        x = self.fc3(x)
        return x

model = SimpleCNN()
model

SimpleCNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1))
  (flat): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=10816, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=6, bias=True)
)

In [49]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

In [52]:
for epoch in range(20):  # Train for 5 epochs
    running_loss = 0.0
    for images, labels in train_loader:
        # Move images and labels to the device

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f'Epoch [{epoch+1}/5], Loss: {running_loss / len(train_loader):.4f}')
     


Epoch [1/5], Loss: 0.9727
Epoch [2/5], Loss: 0.9204
Epoch [3/5], Loss: 0.9040
Epoch [4/5], Loss: 0.8700
Epoch [5/5], Loss: 0.8483
Epoch [6/5], Loss: 0.8003
Epoch [7/5], Loss: 0.7430
Epoch [8/5], Loss: 0.7035
Epoch [9/5], Loss: 0.6788
Epoch [10/5], Loss: 0.6243
Epoch [11/5], Loss: 0.6196
Epoch [12/5], Loss: 0.5597
Epoch [13/5], Loss: 0.5020
Epoch [14/5], Loss: 0.5133
Epoch [15/5], Loss: 0.4520
Epoch [16/5], Loss: 0.4316
Epoch [17/5], Loss: 0.3954
Epoch [18/5], Loss: 0.3236
Epoch [19/5], Loss: 0.3387
Epoch [20/5], Loss: 0.2783


In [53]:
correct = 0
total = 0
with torch.no_grad():  # Disable gradient calculation for evaluation
    for images, labels in test_loader:
        # Move images and labels to the device

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy: {100 * correct / total:.2f}%')
     


Accuracy: 48.03%
